In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.utils import resample
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('../data/raw/train.csv')
val = pd.read_csv('../data/raw/validation.csv')
test= pd.read_csv('../data/raw/test_public.csv')
print(f"Train: {train.shape} Val: {val.shape} Test: {test.shape}")

Train: (180371, 43) | Val: (38651, 43) | Test: (38651, 43)


In [3]:
all_text = pd.concat([train[['proto','service','state']],
                      val[['proto','service','state']],
                      test[['proto','service','state']]])
enc = {}
for col in ['proto', 'service', 'state']:
    le = LabelEncoder()
    le.fit(all_text[col].astype(str))
    enc[col] = le

In [4]:
def build_features(df):
    d = df.copy()
    for col in ['proto', 'service', 'state']:
        d[col + '_enc'] = enc[col].transform(d[col].astype(str))
    d['bytes_ratio']= d['sbytes'] / (d['dbytes'] + 1)
    d['pkts_ratio'] = d['spkts']  / (d['dpkts']  + 1)
    d['load_ratio'] = d['sload']  / (d['dload']  + 1)
    d['total_bytes']= d['sbytes'] + d['dbytes']
    d['total_pkts'] = d['spkts']  + d['dpkts']
    d['bytes_per_src'] = d['sbytes'] / (d['spkts']  + 1)
    d['bytes_per_dst'] = d['dbytes'] / (d['dpkts']  + 1)
    d['loss_ratio'] = (d['sloss'] + d['dloss']) / (d['total_pkts'] + 1)
    d['ttl_diff'] = d['sttl']   - d['dttl']
    d['ttl_sum'] = d['sttl']   + d['dttl']
    d['tcp_setup']= d['synack'] + d['ackdat']
    d['win_ratio'] = d['swin']   / (d['dwin']   + 1)
    d['mean_ratio']= d['smean']  / (d['dmean']  + 1)
    d['pkt_size_diff'] = d['smean']  - d['dmean']
    d['inpkt_ratio'] = d['sinpkt'] / (d['dinpkt'] + 1)
    d['dur_per_pkt']= d['dur']    / (d['total_pkts'] + 1)
    d['ct_total']= (d['ct_srv_src'] + d['ct_srv_dst'] +
                          d['ct_dst_ltm'] + d['ct_src_ltm'])
    d['ct_ratio'] = d['ct_srv_src'] / (d['ct_srv_dst'] + 1)
    d['http_flag'] = (d['ct_flw_http_mthd'] > 0).astype(int)
    d['http_gt3'] = (d['ct_flw_http_mthd'] > 3).astype(int)
    d['dbytes_zero'] = (d['dbytes'] == 0).astype(int)
    d['resp_log']      = np.log1p(d['response_body_len'])
    for col in ['rate','sbytes','dbytes','sload','dload',
                'total_bytes','ct_src_ltm','ct_srv_dst','dur']:
        d[col + '_log'] = np.log1p(d[col])
    cols_drop = ['proto','service','state']
    if 'Label' in d.columns: cols_drop.append('Label')
    if 'id'    in d.columns: cols_drop.append('id')
    return d.drop(columns=cols_drop)

Xtr  = build_features(train); ytr  = train['Label']
Xval = build_features(val);   yval = val['Label']
Xte  = build_features(test)
print(f"Features: {Xtr.shape[1]}")

Features: 73


In [5]:
def oversample(X, y, cibles):
    df = X.copy(); df['_l'] = y.values
    frames = []
    for lbl, g in df.groupby('_l'):
        target = cibles.get(lbl, len(g))
        if len(g) < target:
            g = resample(g, replace=True, n_samples=target, random_state=42)
        frames.append(g)
    bal = pd.concat(frames).sample(frac=1, random_state=42)
    return bal.drop(columns='_l'), bal['_l']

cibles = {0:65100,1:41209,2:31167,3:16972,
          4:11447,5:9791,6:6000,7:6000,8:5000,9:3000}
Xtr_bal, ytr_bal = oversample(Xtr, ytr, cibles)

# Entraînement
lgb_train = lgb.Dataset(Xtr_bal, label=ytr_bal)
lgb_val   = lgb.Dataset(Xval, label=yval, reference=lgb_train)

params = {
    'objective':'multiclass','num_class':10,
    'metric':'multi_logloss','learning_rate':0.05,
    'num_leaves':127,'min_child_samples':20,
    'subsample':0.8,'colsample_bytree':0.8,
    'class_weight':'balanced','random_state':42,
    'n_jobs':-1,'verbose':-1,
}

modele = lgb.train(
    params, lgb_train, num_boost_round=1000,
    valid_sets=[lgb_val],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(50)]
)

probas = modele.predict(Xval)
boosts = np.ones(10)

for _ in range(6):
    change = False
    for ci in range(10):
        best_f1, best_v = 0, boosts[ci]
        for v in [0.3,0.5,0.7,1.0,1.5,2.0,3.0,4.0,5.0]:
            b = boosts.copy(); b[ci] = v
            f1 = f1_score(yval, np.argmax(probas*b, axis=1), average='macro')
            if f1 > best_f1: best_f1=f1; best_v=v
        if best_v != boosts[ci]: change=True
        boosts[ci] = best_v
    f1_cur = f1_score(yval, np.argmax(probas*boosts, axis=1), average='macro')
    print(f"F1={f1_cur:.4f} | boosts={np.round(boosts,1)}")
    if not change: break

print(f"\nScore Val F1-macro FINAL : {f1_score(yval, np.argmax(probas*boosts, axis=1), average='macro'):.4f}")

[50]	valid_0's multi_logloss: 0.437451
[100]	valid_0's multi_logloss: 0.413241
[150]	valid_0's multi_logloss: 0.415679
F1=0.6504 | boosts=[0.7 0.7 0.7 1.  2.  0.3 0.3 1.5 1.  1. ]
F1=0.6506 | boosts=[0.5 1.5 0.7 1.  2.  0.3 0.3 1.5 1.  1. ]
F1=0.6506 | boosts=[0.5 1.5 0.7 1.  2.  0.3 0.3 1.5 1.  1. ]

Score Val F1-macro FINAL : 0.6506
